In [1]:
import sys
import numpy as np
import pandas as pd
import wfdb
import neurokit2 as nk

print("Python:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("WFDB:", wfdb.__version__)
print("NeuroKit:", nk.__version__)


Python: /Users/nikitapatil/CardioSense-AI/venv/bin/python
NumPy: 1.26.4
Pandas: 2.0.3
WFDB: 4.1.2
NeuroKit: 0.2.7


In [1]:
import os
import wfdb
import neurokit2 as nk
import numpy as np
import pandas as pd

FS = 360
WINDOW_SEC = 60
WINDOW_SAMPLES = FS * WINDOW_SEC

ecg_features_list = []
processed = 0
skipped = 0

for rec_id in range(100, 105):
    print(f"\nProcessing record {rec_id}")

    record = wfdb.rdrecord(f"/Users/nikitapatil/Documents/CardioSense-AI/data/mitdb/{rec_id}")
    signal = record.p_signal[:, 0]

    for start in range(0, len(signal) - WINDOW_SAMPLES, WINDOW_SAMPLES):
        segment = signal[start:start + WINDOW_SAMPLES]

        try:
            # ECG processing
            signals, info = nk.ecg_process(segment, sampling_rate=FS)

            # ✅ IMPORTANT: use peaks dictionary, not RR
            peaks = info["ECG_R_Peaks"]

            if np.sum(peaks) < 10:
                skipped += 1
                continue

            # ✅ CORRECT HRV call
            hrv = nk.hrv_time(peaks, sampling_rate=FS)

            if hrv.empty:
                skipped += 1
                continue

            # Record-level label
            ann = wfdb.rdann(f"/Users/nikitapatil/Documents/CardioSense-AI/data/mitdb/{rec_id}", "atr")
            abnormal = {"V", "A", "L", "R", "F", "!", "E"}
            label = int(any(sym in abnormal for sym in ann.symbol))

            hrv["target"] = label
            hrv["record"] = rec_id

            ecg_features_list.append(hrv)
            processed += 1

        except Exception as e:
            skipped += 1

print("\nSUMMARY")
print("Processed windows:", processed)
print("Skipped windows:", skipped)



Processing record 100

Processing record 101

Processing record 102

Processing record 103

Processing record 104

SUMMARY
Processed windows: 150
Skipped windows: 0


In [1]:
import os
import wfdb
import neurokit2 as nk
import numpy as np
import pandas as pd

FS = 360
WINDOW_SEC = 60
WINDOW_SAMPLES = FS * WINDOW_SEC

# ✅ Balanced record selection
NORMAL_RECORDS = [114, 115, 116, 117]
ABNORMAL_RECORDS = [200, 201, 202, 203, 205]

record_ids = NORMAL_RECORDS + ABNORMAL_RECORDS

ecg_features_list = []

for rec_id in record_ids:
    print(f"Processing record {rec_id}")

    record = wfdb.rdrecord(f"/Users/nikitapatil/Documents/CardioSense-AI/data/mitdb/{rec_id}")
    signal = record.p_signal[:, 0]

    # Record-level label
    ann = wfdb.rdann(f"../data/mitdb/{rec_id}", "atr")
    abnormal_symbols = {"V", "A", "L", "R", "F", "!", "E"}
    label = int(any(sym in abnormal_symbols for sym in ann.symbol))

    for start in range(0, len(signal) - WINDOW_SAMPLES, WINDOW_SAMPLES):
        segment = signal[start:start + WINDOW_SAMPLES]

        try:
            signals, info = nk.ecg_process(segment, sampling_rate=FS)
            peaks = info["ECG_R_Peaks"]

            if np.sum(peaks) < 10:
                continue

            # Correct HRV extraction
            hrv = nk.hrv_time(peaks, sampling_rate=FS)

            if hrv.empty:
                continue

            hrv["target"] = label
            hrv["record"] = rec_id

            ecg_features_list.append(hrv)

        except Exception:
            continue

# Combine
ecg_df = pd.concat(ecg_features_list, ignore_index=True)

print("Final ECG dataset shape:", ecg_df.shape)
print("Class distribution:")
print(ecg_df["target"].value_counts())

# Save
os.makedirs("/Users/nikitapatil/Documents/CardioSense-AI/data/mitdb", exist_ok=True)
ecg_df.to_csv("/Users/nikitapatil/Documents/CardioSense-AI/data/mitdb/ecg_features.csv", index=False)

print("ecg_features.csv saved successfully")


Processing record 114
Processing record 115
Processing record 116
Processing record 117
Processing record 200
Processing record 201
Processing record 202
Processing record 203
Processing record 205
Final ECG dataset shape: (270, 27)
Class distribution:
target
1    240
0     30
Name: count, dtype: int64
ecg_features.csv saved successfully


In [2]:
ecg_df = pd.concat(ecg_features_list, ignore_index=True)
ecg_df.shape


(150, 27)

In [4]:
os.makedirs("/Users/nikitapatil/Documents/CardioSense-AI/data/mitdb", exist_ok=True)
ecg_df.to_csv("/Users/nikitapatil/Documents/CardioSense-AI/data/mitdb/ecg_features.csv", index=False)
print("ecg_features.csv saved successfully")


ecg_features.csv saved successfully
